---
**© 2026 Nicola Vessio — Tutti i diritti riservati.**

Data di creazione dell'opera: **7 agosto 2026**

Questo software (notebook, codice e relativa documentazione) è opera di **Nicola Vessio**
ed è protetto dalle norme vigenti in materia di diritto d'autore (L. 633/1941 e successive
modifiche, Convenzione di Berna, Direttiva 2009/24/CE sulla tutela giuridica dei programmi
per elaboratore).

Sono riservati all'autore tutti i diritti di utilizzazione economica dell'opera, inclusi
a titolo esemplificativo la riproduzione, la distribuzione, la modifica, l'adattamento,
la traduzione e la comunicazione al pubblico. **Ogni uso non espressamente autorizzato
per iscritto dall'autore è vietato.**

_Contatto: da definire._

---

# 📊 REPORT DELLE OPERAZIONI MT5 - AUSILIO ALLA DICHIARAZIONE DEI REDDITI

Il presente notebook scarica lo storico delle operazioni dal terminale **MetaTrader 5**
(aperto e loggato sul PC) relative a un **periodo definito dall'utente**, allo scopo di
produrre un report chiaro e ordinato **a supporto della dichiarazione dei redditi**.

## Conti supportati
Lo strumento gestisce **più conti MetaTrader 5 distinti**, anche presso broker esteri
differenti (ad esempio **TMGM** — TradeMax Global — e **FPG** — Fortune Prime Global).

L'analisi avviene su **un conto alla volta**: il notebook si collega al terminale MT5
attualmente aperto e loggato. Per ottenere il report di un conto specifico si tiene aperto
**un solo terminale per volta** (quello del broker da analizzare), si esegue il notebook, e
si ripete eventualmente con l'altro. All'avvio vengono sempre verificati i dati del conto
(broker e numero di login) per assicurarsi di analizzare quello corretto.

## Funzionalità
- Scaricare i *deal* (eventi di esecuzione) del conto per un intervallo di date definito
- Ricucire i deal nelle **operazioni complete** (entrata + uscita) con il relativo profitto/perdita
- Calcolare metriche di sintesi e produrre un **report Excel** consultabile dal commercialista

## 💱 Valuta del conto
I conti considerati sono denominati in **dollari statunitensi (USD)**. Di conseguenza
**tutti gli importi** riportati in questo notebook (saldo, profitti, perdite, commissioni,
ecc.) sono espressi **in USD**. La **conversione in euro (EUR)** ai fini della dichiarazione
dei redditi italiana **non** è gestita qui ed è a cura dello studio commercialistico.

## Prerequisiti (per rieseguire il notebook)
- Windows con il **terminale MT5 installato, aperto e loggato** sul conto da analizzare
- Un solo terminale MT5 aperto per volta (per evitare ambiguità sul conto)
- Pacchetto ufficiale `MetaTrader5` (verificare versione `5.0.xxxx`)
- Il terminale va tenuto aperto **solo durante l'esecuzione** dello script

## ⚠️ Nota importante
Questo notebook è uno **strumento di organizzazione ed esposizione dei dati**, non una
consulenza fiscale. I dati vanno **verificati** e la dichiarazione è gestita dal
commercialista o CAF. Il regime fiscale applicabile, le eventuali compensazioni, la
corretta imputazione di plusvalenze e minusvalenze, la conversione in euro, il quadro RW
e l'imposta di bollo sono di competenza dello studio commercialistico.

In [ ]:
# ============================================================
# IMPOSTAZIONI — Periodo da analizzare
# ============================================================
# Unico punto in cui si definisce l'intervallo di date del report.
# Cambiando SOLO queste due righe si cambia il periodo per l'intero notebook.
# Formato: datetime(ANNO, MESE, GIORNO) oppure datetime(ANNO, MESE, GIORNO, ORA, MINUTO, SECONDO)

from datetime import datetime

data_inizio = datetime(2026, 1, 1)                 # primo giorno dell'anno fiscale da analizzare
data_fine   = datetime(2026, 12, 31, 23, 59, 59)   # ultimo istante dell'anno fiscale da analizzare

print(f"Periodo impostato: dal {data_inizio:%d/%m/%Y} al {data_fine:%d/%m/%Y}")

In [ ]:
# ============================================================
# Connessione al terminale MT5 e verifica del conto
# ============================================================
# Questa cella apre il "ponte" tra Python e il terminale MetaTrader 5
# che deve essere già aperto e loggato sul PC. Serve a controllare
# che la connessione funzioni e a leggere i dati anagrafici del conto.

import MetaTrader5 as mt5   # libreria ufficiale per dialogare con il terminale MT5

# initialize() aggancia Python al terminale MT5 aperto.
# Se restituisce False, la connessione è fallita: stampiamo il motivo e fermiamo.
if not mt5.initialize():
    print("Connessione fallita:", mt5.last_error())  # last_error() spiega perché
    raise SystemExit("Interrompo: terminale MT5 non raggiungibile.")

# account_info() restituisce i dati del conto attualmente loggato nel terminale.
conto = mt5.account_info()

# Stampa DISCRETA: mostriamo solo ciò che serve a confermare di essere sul conto
# giusto (broker e valuta), con il numero di login MASCHERATO (solo ultime 3 cifre).
# Evitiamo di esporre login completo e saldo negli output del notebook.
login_mascherato = f"***{str(conto.login)[-3:]}" if conto else "n/d"
print("Connessione riuscita.")
print(f"  Broker:  {conto.company}")
print(f"  Server:  {conto.server}")
print(f"  Valuta:  {conto.currency}")
print(f"  Login:   {login_mascherato}")

# shutdown() chiude in modo pulito il ponte con il terminale quando abbiamo finito.
mt5.shutdown()

In [ ]:
# ============================================================
# Scarico dei "deal" (operazioni) per il periodo scelto
# ============================================================
# In MT5 lo storico è composto da "deal": singoli eventi di esecuzione.
# Un trade completo (apertura + chiusura) è formato da DUE deal:
#   - un deal di ENTRATA (IN)
#   - un deal di USCITA (OUT), a cui è associato il profitto/perdita
# In questa cella scarichiamo tutti i deal del periodo e ne osserviamo uno,
# per capire quali informazioni abbiamo a disposizione.

import MetaTrader5 as mt5

# Riapriamo il ponte con il terminale (ogni esecuzione è indipendente).
if not mt5.initialize():
    print("Connessione fallita:", mt5.last_error())
    raise SystemExit("Interrompo: terminale MT5 non raggiungibile.")

# Le date arrivano dalla cella IMPOSTAZIONI (data_inizio, data_fine)
# history_deals_get() scarica TUTTI i deal compresi tra le due date.
deals = mt5.history_deals_get(data_inizio, data_fine)

# MT5, in caso di problemi, restituisce None invece di segnalare un errore:
# per questo controlliamo esplicitamente i tre casi possibili.
if deals is None:
    # None = errore nella richiesta (es. periodo non valido, terminale non pronto)
    print("Nessun deal o errore:", mt5.last_error())
elif len(deals) == 0:
    # Lista vuota = richiesta ok, ma nessuna operazione in quel periodo
    print("Zero deal nel periodo scelto.")
else:
    # Almeno un deal trovato: stampiamo quanti e mostriamo il primo come esempio
    print(f"Trovati {len(deals)} deal.")
    print(deals[0])   # struttura di un singolo deal: utile per costruire l'analisi

# Chiudiamo il ponte con il terminale.
mt5.shutdown()

In [ ]:
# ============================================================
# Classificazione dei deal: trade veri vs movimenti di cassa
# ============================================================
# Non tutti i deal sono operazioni di trading. In MT5 il campo "type" distingue:
#   type = 0  -> BUY  (apertura al rialzo: parte di un trade)
#   type = 1  -> SELL (apertura al ribasso: parte di un trade)
#   type = 2  -> BALANCE (deposito, prelievo o correzione di saldo: NON è un trade)
# Ai fini fiscali i movimenti di cassa (depositi/prelievi) vanno tenuti separati
# dai profitti/perdite generati dalle operazioni di trading.

import MetaTrader5 as mt5

if not mt5.initialize():
    print("Connessione fallita:", mt5.last_error())
    raise SystemExit("Interrompo: terminale MT5 non raggiungibile.")

# Le date arrivano dalla cella IMPOSTAZIONI (data_inizio, data_fine)
deals = mt5.history_deals_get(data_inizio, data_fine)
mt5.shutdown()

if not deals:   # copre sia None sia lista vuota
    raise SystemExit("Nessun deal scaricato: controlla terminale e periodo.")

# Separiamo i deal in due gruppi in base al campo "type".
movimenti_cassa = [d for d in deals if d.type == 2]           # depositi/prelievi
deal_trading    = [d for d in deals if d.type in (0, 1)]      # buy e sell (trade)

# Riepilogo di controllo
print(f"Deal totali:            {len(deals)}")
print(f"Movimenti di cassa:     {len(movimenti_cassa)}  (depositi/prelievi)")
print(f"Deal di trading:        {len(deal_trading)}  (buy/sell)")

# Vediamo un esempio di deal di trading vero (il primo del gruppo)
if deal_trading:
    print("\nEsempio di deal di trading:")
    print(deal_trading[0])

In [ ]:
# ============================================================
# Ricucitura dei deal in trade completi (per posizione)
# ============================================================
# Ogni trade è identificato da un "position_id": tutti i deal con lo stesso
# position_id appartengono alla stessa operazione. All'interno di una posizione:
#   entry = 0  -> deal di APERTURA (ingresso a mercato)
#   entry = 1  -> deal di CHIUSURA (uscita: porta profit, swap e commissioni finali)
# Il profitto netto reale di un trade è dato da: profit + swap + commission
# (swap e commission sono tipicamente costi, quindi valori negativi).

import pandas as pd
from datetime import datetime

# Raggruppiamo i deal di trading per position_id
posizioni = {}
for d in deal_trading:
    pid = d.position_id
    if pid not in posizioni:
        posizioni[pid] = []
    posizioni[pid].append(d)

# Ricostruiamo un trade completo per ogni posizione
trade_completi = []
for pid, lista_deal in posizioni.items():
    # Ordiniamo i deal della posizione per tempo (prima l'apertura, poi la chiusura)
    lista_deal.sort(key=lambda x: x.time)

    apertura = lista_deal[0]     # primo deal = apertura
    chiusura = lista_deal[-1]    # ultimo deal = chiusura

    # Sommiamo le componenti economiche su tutti i deal della posizione.
    # Queste tre variabili vivono solo dentro questo ciclo:
    profit_totale     = sum(d.profit for d in lista_deal)      # utile/perdita lordo
    swap_totale       = sum(d.swap for d in lista_deal)        # costi/proventi overnight
    commission_totale = sum(d.commission for d in lista_deal)  # commissioni broker

    trade_completi.append({
        "position_id":   pid,
        "symbol":        apertura.symbol,                          # strumento (es. EURUSD)
        "volume":        apertura.volume,                          # lotti dell'apertura
        "tipo":          "BUY" if apertura.type == 0 else "SELL",  # direzione iniziale
        "data_apertura": datetime.fromtimestamp(apertura.time),    # convertiamo il timestamp in data
        "data_chiusura": datetime.fromtimestamp(chiusura.time),
        "profit_lordo":  round(profit_totale, 2),
        "swap":          round(swap_totale, 2),
        "commission":    round(commission_totale, 2),
        # Profitto netto dei costi del broker (commissioni + swap), ma lordo di imposte.
        # In USD. È il risultato realmente rimasto dopo i costi, tasse escluse.
        "profit_netto_costi_broker_lordo_imposte_USD":
            round(profit_totale + swap_totale + commission_totale, 2),
    })

# Raccogliamo tutto in una tabella ordinata per data di chiusura
df_trade = pd.DataFrame(trade_completi).sort_values("data_chiusura").reset_index(drop=True)

print(f"Trade completi ricostruiti: {len(df_trade)}")
print(f"Profitto netto totale (USD): {df_trade['profit_netto_costi_broker_lordo_imposte_USD'].sum():.2f}")
print()
print(df_trade.head(10))

In [ ]:
# ============================================================
# Generazione del report Excel per il commercialista
# ============================================================
# Creiamo un file .xlsx con QUATTRO fogli:
#   1) Operazioni        -> una riga per ogni trade (le posizioni chiuse)
#   2) Riepilogo         -> dati del conto + totali fiscali (con FORMULE Excel vere)
#   3) Movimenti di cassa-> depositi/prelievi, tenuti separati dai trade
#   4) Guida e glossario -> spiegazioni (BUY/SELL, forex, termini) a corredo
# I totali del Riepilogo sono formule: se si modifica una riga, si ricalcolano da soli.

import pandas as pd
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# --- Leggiamo i dati del conto (servono per l'intestazione e per il nome file) ---
import MetaTrader5 as mt5
mt5.initialize()
info = mt5.account_info()
mt5.shutdown()

# --- Prepariamo i dati dei movimenti di cassa in una tabella ordinata ---
# (movimenti_cassa proviene dalla cella di classificazione: sono i deal con type == 2)
mov_cassa = []
for d in movimenti_cassa:
    mov_cassa.append({
        "ticket":   d.ticket,
        "data":     datetime.fromtimestamp(d.time),
        "importo":  round(d.profit, 2),   # per i balance, 'profit' è l'importo versato/prelevato
        "tipo":     "Deposito" if d.profit > 0 else "Prelievo",
        "commento": d.comment,
    })
df_cassa = pd.DataFrame(mov_cassa).sort_values("data").reset_index(drop=True)

# --- Stili riutilizzabili ---
BLU        = "1F4E78"   # intestazioni
GRIGIO     = "D9D9D9"   # righe totali
ZEBRA      = "EEF3F8"   # azzurrino tenue per le righe alternate
VERDE_TXT  = "1E7B34"   # testo verde per profitti positivi
ROSSO_TXT  = "C0392B"   # testo rosso per profitti negativi
font_tit   = Font(name="Arial", size=14, bold=True, color="1F4E78")
font_head  = Font(name="Arial", size=11, bold=True, color="FFFFFF")
font_norm  = Font(name="Arial", size=10)
font_bold  = Font(name="Arial", size=10, bold=True)
font_verde = Font(name="Arial", size=10, color=VERDE_TXT)
font_rosso = Font(name="Arial", size=10, color=ROSSO_TXT)
font_verde_b = Font(name="Arial", size=10, bold=True, color=VERDE_TXT)
font_rosso_b = Font(name="Arial", size=10, bold=True, color=ROSSO_TXT)
fill_head  = PatternFill("solid", fgColor=BLU)
fill_tot   = PatternFill("solid", fgColor=GRIGIO)
fill_zebra = PatternFill("solid", fgColor=ZEBRA)
bordo      = Border(*[Side(style="thin", color="BFBFBF")]*4)
centro     = Alignment(horizontal="center", vertical="center")
sinistra   = Alignment(horizontal="left", vertical="top", wrap_text=True)
sx_centro  = Alignment(horizontal="left", vertical="center")

def scrivi_intestazioni(ws, intestazioni, riga=1):
    """Scrive una riga di intestazioni con lo stile blu."""
    for col, testo in enumerate(intestazioni, start=1):
        c = ws.cell(row=riga, column=col, value=testo)
        c.font = font_head
        c.fill = fill_head
        c.alignment = centro
        c.border = bordo

def banda_titolo(ws, riga, testo, n_colonne):
    """Scrive un titolo di sezione su banda blu, esteso su n_colonne."""
    for col in range(1, n_colonne + 1):
        c = ws.cell(row=riga, column=col)
        c.fill = fill_head
        c.border = bordo
        if col == 1:
            c.value = testo
            c.font = font_head
            c.alignment = sx_centro

# ============================================================
# FOGLIO 1 — OPERAZIONI
# ============================================================
wb = Workbook()
ws1 = wb.active
ws1.title = "Operazioni"

intest_op = [
    "N.", "ID Posizione", "Strumento", "Volume (lotti)", "Tipo",
    "Data apertura", "Data chiusura",
    "Profit lordo (USD)", "Swap (USD)", "Commissione (USD)",
    "Profit netto costi broker - lordo imposte (USD)",
]
scrivi_intestazioni(ws1, intest_op)

for i, (_, r) in enumerate(df_trade.iterrows(), start=1):
    riga = i + 1
    netto = r["profit_netto_costi_broker_lordo_imposte_USD"]
    riga_colorata = (i % 2 == 0)
    valori = [
        i, r["position_id"], r["symbol"], r["volume"], r["tipo"],
        r["data_apertura"].strftime("%d/%m/%Y %H:%M:%S"),
        r["data_chiusura"].strftime("%d/%m/%Y %H:%M:%S"),
        r["profit_lordo"], r["swap"], r["commission"], netto,
    ]
    for col, v in enumerate(valori, start=1):
        c = ws1.cell(row=riga, column=col, value=v)
        c.font = font_norm
        c.border = bordo
        if riga_colorata:
            c.fill = fill_zebra
        if col in (8, 9, 10, 11):
            c.number_format = '#,##0.00'
        if col == 11:
            c.font = font_verde if netto >= 0 else font_rosso

ultima_riga_op = len(df_trade) + 1
ws1.freeze_panes = "A2"

larghezze_op = [5, 14, 11, 13, 8, 20, 20, 16, 12, 14, 30]
for col, w in enumerate(larghezze_op, start=1):
    ws1.column_dimensions[get_column_letter(col)].width = w

# ============================================================
# FOGLIO 2 — RIEPILOGO (dati del conto + totali con formule)
# ============================================================
ws2 = wb.create_sheet("Riepilogo")

col_netto = "K"
rng_netto = f"Operazioni!{col_netto}2:{col_netto}{ultima_riga_op}"

# --- Banda titolo: DATI DEL CONTO ---
banda_titolo(ws2, 1, "DATI DEL CONTO", 2)

dati_conto = [
    ("Intestatario",       info.name     if info else "n/d"),
    ("Broker",             info.company  if info else "n/d"),
    ("Numero conto",       str(info.login)    if info else "n/d"),
    ("Server",             info.server   if info else "n/d"),
    ("Valuta",             info.currency if info else "n/d"),
    ("Periodo analizzato", f"dal {data_inizio:%d/%m/%Y} al {data_fine:%d/%m/%Y}"),
]

r = 2
for i, (etichetta, valore) in enumerate(dati_conto):
    ce = ws2.cell(row=r, column=1, value=etichetta)
    cv = ws2.cell(row=r, column=2, value=valore)
    ce.font = font_bold
    cv.font = font_norm
    ce.border = bordo
    cv.border = bordo
    if i % 2 == 1:   # zebra
        ce.fill = fill_zebra
        cv.fill = fill_zebra
    r += 1

# --- Banda titolo: RIEPILOGO FISCALE ---
r += 1   # riga vuota di stacco
banda_titolo(ws2, r, "RIEPILOGO FISCALE", 2)
r += 1

righe_riep = [
    ("Numero di operazioni (trade)",        f'=COUNT({rng_netto})'),
    ("Operazioni in guadagno (plus)",       f'=COUNTIF({rng_netto},">0")'),
    ("Operazioni in perdita (minus)",       f'=COUNTIF({rng_netto},"<0")'),
    ("Somma plusvalenze nette (USD)",       f'=SUMIF({rng_netto},">0")'),
    ("Somma minusvalenze nette (USD)",      f'=SUMIF({rng_netto},"<0")'),
    ("RISULTATO NETTO COMPLESSIVO (USD)",   f'=SUM({rng_netto})'),
    ("Totale commissioni (USD)",            f'=SUM(Operazioni!J2:J{ultima_riga_op})'),
    ("Totale swap (USD)",                   f'=SUM(Operazioni!I2:I{ultima_riga_op})'),
]

for i, (etichetta, formula) in enumerate(righe_riep):
    ce = ws2.cell(row=r, column=1, value=etichetta)
    c  = ws2.cell(row=r, column=2, value=formula)
    ce.font = font_bold
    c.font = font_norm
    c.number_format = '#,##0.00'
    ce.border = bordo
    c.border = bordo
    if "COMPLESSIVO" in etichetta:
        # Riga chiave evidenziata in grigio, valore in grassetto
        ce.fill = fill_tot
        c.fill = fill_tot
        c.font = font_bold
    elif i % 2 == 1:
        ce.fill = fill_zebra
        c.fill = fill_zebra
    r += 1

# Nota valuta
r += 1
ws2.cell(row=r, column=1,
         value="Tutti gli importi sono in USD. Conversione in EUR, quadro RW e "
               "imposta di bollo a cura dello studio commercialistico.").font = font_norm

ws2.column_dimensions["A"].width = 42
ws2.column_dimensions["B"].width = 26

# ============================================================
# FOGLIO 3 — MOVIMENTI DI CASSA
# ============================================================
ws3 = wb.create_sheet("Movimenti di cassa")
ws3["A1"] = ("Depositi e prelievi — NON sono operazioni di trading, "
             "non generano plusvalenze/minusvalenze")
ws3["A1"].font = font_bold

intest_cassa = ["Ticket", "Data", "Importo (USD)", "Tipo", "Commento"]
scrivi_intestazioni(ws3, intest_cassa, riga=3)

for i, (_, r_) in enumerate(df_cassa.iterrows(), start=1):
    riga = i + 3
    riga_colorata = (i % 2 == 0)
    valori = [
        r_["ticket"],
        r_["data"].strftime("%d/%m/%Y %H:%M:%S"),
        r_["importo"], r_["tipo"], r_["commento"],
    ]
    for col, v in enumerate(valori, start=1):
        c = ws3.cell(row=riga, column=col, value=v)
        c.font = font_norm
        c.border = bordo
        if riga_colorata:
            c.fill = fill_zebra
        if col == 3:
            c.number_format = '#,##0.00'

ws3.freeze_panes = "A4"

for col, w in enumerate([12, 20, 14, 12, 25], start=1):
    ws3.column_dimensions[get_column_letter(col)].width = w

# ============================================================
# FOGLIO 4 — GUIDA E GLOSSARIO
# ============================================================
ws4 = wb.create_sheet("Guida e glossario")

# --- Sezione discorsiva: COME LEGGERE QUESTO REPORT ---
banda_titolo(ws4, 1, "COME LEGGERE QUESTO REPORT", 2)

paragrafi = [
    'Questo report riguarda operazioni di trading su strumenti finanziari derivati (CFD) '
    'effettuate tramite due broker esteri (ad esempio TMGM - TradeMax Global e FPG - Fortune Prime Global).',
    '"BUY" e "SELL" NON sono acquisti o vendite di beni o valute reali. Sono contratti (CFD) '
    'che replicano la variazione di prezzo di uno strumento: non si possiede mai il bene o la '
    'valuta sottostante. BUY = puntata sul rialzo (posizione lunga); SELL = puntata sul ribasso '
    '(posizione corta). Il risultato è solo un importo in denaro (USD).',
    'FOREX (coppie di valute): strumenti come EURUSD, GBPUSD, USDJPY. Si opera sulla variazione '
    'del tasso di cambio, non sullo scambio fisico delle valute. XAUUSD indica oro/dollaro, '
    'anch\'esso trattato come strumento e mai consegnato fisicamente.',
    'VALUTA: il conto è denominato in dollari USA (USD). Tutti gli importi sono in USD. '
    'La conversione in euro è a cura dello studio commercialistico.',
    'BROKER ESTERI: quando il broker ha sede estera, gli adempimenti collegati '
    '(quadro RW, imposta di bollo) sono curati dallo studio commercialistico.',
]

r = 3
for p in paragrafi:
    c = ws4.cell(row=r, column=1, value=p)
    c.font = font_norm
    c.alignment = sinistra
    ws4.merge_cells(start_row=r, start_column=1, end_row=r, end_column=2)
    ws4.row_dimensions[r].height = 45
    r += 2

# --- Sezione tabellare: GLOSSARIO DEI TERMINI (due colonne) ---
r += 1
banda_titolo(ws4, r, "GLOSSARIO DEI TERMINI", 2)
r += 1
# intestazione tabella
scrivi_intestazioni(ws4, ["Termine", "Significato"], riga=r)
r += 1

glossario = [
    ("CFD", "Contratto che replica il prezzo di uno strumento senza possederlo. Risultato in denaro."),
    ("Forex", "Mercato dei tassi di cambio, strumenti espressi come coppie di valute."),
    ("Coppia di valute", "Es. EURUSD: prima valuta 'base', seconda 'quotata'. Si opera sul cambio."),
    ("BUY (posizione lunga)", "Puntata sul rialzo del prezzo. Non è acquisto di beni/valute reali."),
    ("SELL (posizione corta)", "Puntata sul ribasso del prezzo. Non è vendita di beni/valute reali."),
    ("Deal", "Singolo evento registrato dalla piattaforma (entrata, uscita o movimento di cassa)."),
    ("Trade (operazione)", "Operazione completa: entrata + uscita. Composta da più deal."),
    ("Position ID", "Codice che identifica una singola operazione, collega entrata e uscita."),
    ("Volume (lotti)", "Dimensione dell'operazione. Non indica quantità di merce/valuta posseduta."),
    ("Symbol (strumento)", "Nome dello strumento (es. EURUSD, GBPUSD, XAUUSD)."),
    ("Profit lordo", "Risultato dell'operazione prima dei costi del broker. In USD."),
    ("Commission", "Costo trattenuto dal broker. Valore negativo. In USD."),
    ("Swap", "Interesse per il mantenimento della posizione oltre la giornata. Di norma negativo. In USD."),
    ("Profit netto costi broker", "Risultato dopo commissioni e swap. Al lordo delle imposte. In USD."),
    ("Balance (movimento di cassa)", "Deposito o prelievo. Non è trading, non genera plus/minusvalenze."),
    ("Plusvalenza", "Guadagno complessivo realizzato."),
    ("Minusvalenza", "Perdita complessiva realizzata."),
    ("Quadro RW", "Sezione per il monitoraggio delle attività finanziarie estere. A cura dello studio."),
    ("Imposta di bollo", "Imposta annua sui conti oltre soglie di giacenza. A cura dello studio."),
]

for i, (termine, significato) in enumerate(glossario):
    ct = ws4.cell(row=r, column=1, value=termine)
    cs = ws4.cell(row=r, column=2, value=significato)
    ct.font = font_bold
    cs.font = font_norm
    ct.alignment = sx_centro
    cs.alignment = sinistra
    ct.border = bordo
    cs.border = bordo
    if i % 2 == 1:
        ct.fill = fill_zebra
        cs.fill = fill_zebra
    r += 1

ws4.column_dimensions["A"].width = 30
ws4.column_dimensions["B"].width = 85

# ============================================================
# SALVATAGGIO
# ============================================================
company = (info.company if info else "").upper()
if "TRADEMAX" in company:
    broker = "TMGM"
elif "FORTUNE" in company:
    broker = "FPG"
else:
    broker = "BROKER"

anno = data_inizio.year
nome_file = f"report_trading_{broker}_{anno}.xlsx"

wb.save(nome_file)
print(f"Report salvato: {nome_file}")
print(f"  - Broker rilevato: {info.company if info else 'n/d'}")
print(f"  - Operazioni: {len(df_trade)} trade")
print(f"  - Movimenti di cassa: {len(df_cassa)} movimenti")
print("  - IMPORTANTE: aprire e salvare in Excel prima di inviare, per calcolare i totali.")

## 🧾 APPENDICE - COME LEGGERE QUESTO REPORT

Il report riguarda operazioni di **trading su strumenti finanziari derivati (CFD)**
effettuate tramite broker online (ad esempio **TMGM - TradeMax Global** e
**FPG - Fortune Prime Global**). Si chiariscono i punti che generano più spesso confusione:

### ❗ "BUY" e "SELL" NON sono acquisti o vendite di beni o valute reali
Le diciture **BUY** (compra) o **SELL** (vendi) su strumenti come `EURUSD`, `GBPUSD`,
`USDJPY` o `XAUUSD` **non indicano** l'acquisto o la vendita fisica di valuta estera, oro
o altre merci.

Si tratta di **contratti finanziari (CFD - Contract For Difference)**: contratti che
replicano la variazione di prezzo di uno strumento. Il bene o la valuta sottostante **non
viene mai posseduto**. Non vengono ricevuti dollari, sterline, yen o lingotti d'oro: esiste
solo un **profitto o una perdita in denaro** (in USD), in base all'andamento del prezzo.

- **BUY** = posizione al rialzo ("lunga"): si punta sull'aumento del prezzo
- **SELL** = posizione al ribasso ("corta"): si punta sulla diminuzione del prezzo

In entrambi i casi il risultato è **solo un importo in denaro** (in USD), non un bene fisico.

### 💱 Cosa sono le coppie di valute (Forex)
Il **Forex** (Foreign Exchange) è il mercato dei tassi di cambio. Gli strumenti si
presentano come **coppie di valute**, es. `EURUSD`: la prima valuta (EUR) è la "base",
la seconda (USD) è la "quotata".

Operare su una coppia significa **puntare sulla variazione del tasso di cambio**, non
scambiare fisicamente le due valute. Alcuni esempi di coppie:
- `EURUSD` — Euro / Dollaro USA
- `GBPUSD` — Sterlina britannica / Dollaro USA
- `USDJPY` — Dollaro USA / Yen giapponese
- `XAUUSD` — Oro / Dollaro USA

### 💵 Tutti gli importi sono in DOLLARI USA (USD)
Il conto di trading è denominato in **USD**. Ogni cifra del report (profitti, perdite,
commissioni) è espressa **in dollari**. La conversione in euro è a cura dello studio
commercialistico, ai cambi di riferimento.

### 🌍 Broker esteri
Quando i broker hanno sede **estera** (come nel caso di TMGM e FPG), possono sussistere
adempimenti specifici (es. **quadro RW** per il monitoraggio delle attività finanziarie
detenute all'estero, ed eventuale **imposta di bollo**), di competenza dello studio
commercialistico.

### 📌 In sintesi
Non vi è stato **alcun acquisto o vendita di valute o merci**. Sono stati aperti e chiusi
**contratti finanziari** il cui unico esito è stato un **guadagno o una perdita in denaro
(USD)**, riepilogati nel presente documento.